# Options Implied Skew as a Regime Classifier for Equity Factor Crowding
## Exploratory Analysis Notebook

**Research Question:** Can the options market's forward-looking hedging demand (put skew) serve as a leading indicator of equity factor crowding risk?

**Key Hypothesis:** When institutional investors accumulate large positions in a specific equity factor (momentum, low-vol, quality), they simultaneously buy downside protection (OTM puts) on those positions. This drives up the put implied volatility relative to ATM IV — the *put skew*. An elevated cross-sectional median skew across factor constituents signals crowding risk **before** the actual factor unwind.

---

### Notebook Contents
1. Universe Construction
2. Factor Portfolio Construction  
3. Options Skew Data Overview
4. Skew Z-Score Construction
5. Factor Crowding Index Construction
6. Lead-Lag Analysis
7. Regime Classification
8. Crash Date Validation
9. Backtest Results
10. Limitations & Extensions

In [ ]:
# Standard imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# Add project root to path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from src import data_loader, factor_engine, skew_engine, crowding_detector, regime_filter, backtest, visualize

# Configure plotting
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')
    
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = [14, 6]

print(f'Project root: {project_root}')
print(f'Backtest period: {config.BACKTEST_START} → {config.BACKTEST_END}')
print(f'Universe size: {config.UNIVERSE_SIZE} stocks')

---
## Section 1: Universe Construction

We start with the S&P 500 constituent list and select the top 100 stocks by market cap. Using only the top 100 names ensures liquid options markets and reliable skew data.

**Survivorship bias note:** Using the current S&P 500 list introduces survivorship bias — companies that failed or were delisted are excluded. For a rigorous out-of-sample study, use a point-in-time index membership database (e.g., Compustat via WRDS).

In [ ]:
# ── Demonstrate universe construction pipeline ───────────────────────────────────────────────
print('S&P 500 Universe Construction')
print('=' * 50)

# Load sector ETF map to understand sector coverage
sector_map = pd.read_csv(config.SECTOR_ETF_MAP_CSV)
print('\nSector ETF Map:')
print(sector_map.to_string(index=False))

print(f'\nDefensive sectors: {sector_map[sector_map.defensive_flag == True]["sector_name"].tolist()}')
print(f'Cyclical sectors: {sector_map[sector_map.defensive_flag == False]["sector_name"].tolist()}')

In [ ]:
# Demonstrate with synthetic universe data (to avoid live scraping in notebook)
# In production, replace with: tickers = data_loader.fetch_sp500_tickers()

# Synthetic large-cap universe for demonstration
DEMO_TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'LLY', 'UNH',
    'JPM', 'V', 'AVGO', 'XOM', 'MA', 'PG', 'JNJ', 'COST', 'HD', 'ABBV',
    'MRK', 'NFLX', 'WMT', 'CVX', 'CRM', 'BAC', 'AMD', 'ORCL', 'KO', 'PEP',
    'ADBE', 'ACN', 'TMO', 'MCD', 'CSCO', 'NKE', 'ABT', 'CAT', 'GS', 'MS',
    'RTX', 'AMGN', 'TXN', 'INTU', 'LOW', 'PFE', 'SPGI', 'AXP', 'QCOM', 'HON',
    'AMAT', 'DHR', 'ISRG', 'BKNG', 'GE', 'PANW', 'C', 'VRTX', 'LRCX', 'SYK',
    'PLD', 'CB', 'KLAC', 'MO', 'CI', 'MDLZ', 'TJX', 'DE', 'HCA', 'ETN',
    'DUK', 'SO', 'BDX', 'CMG', 'REGN', 'WM', 'SHW', 'APH', 'CL', 'MCO',
    'ZTS', 'CTAS', 'MMM', 'MAR', 'FI', 'ADP', 'NSC', 'USB', 'EOG', 'PCAR',
    'TFC', 'EMR', 'GD', 'IDXX', 'PCG', 'VLO', 'PSX', 'CME', 'MSCI', 'CARR',
    # Defensive ETFs for the basket
    'XLU', 'XLP', 'SPLV', 'VYM',
]

UNIVERSE = [t for t in DEMO_TICKERS if t not in ['XLU', 'XLP', 'SPLV', 'VYM']]
UNIVERSE = UNIVERSE[:100]  # Cap at 100

print(f'Universe size: {len(UNIVERSE)} stocks')
print(f'Sample tickers: {UNIVERSE[:10]}')

In [ ]:
# Sector breakdown (approximate)
sector_assignments = {
    'Information Technology': ['AAPL','MSFT','NVDA','AVGO','AMD','ORCL','ADBE','CRM','CSCO','TXN',
                                 'INTU','AMAT','PANW','LRCX','KLAC','APH'],
    'Financials': ['JPM','V','MA','BAC','GS','MS','AXP','SPGI','C','MCO','CME','MSCI','USB','TFC'],
    'Health Care': ['UNH','LLY','JNJ','ABBV','MRK','TMO','ABT','AMGN','PFE','DHR','ISRG','SYK',
                    'BDX','REGN','CI','VRTX','ZTS','HCA'],
    'Consumer Discretionary': ['AMZN','TSLA','HD','MCD','NKE','LOW','BKNG','TJX','CMG','MAR'],
    'Communication Services': ['GOOGL','META','NFLX'],
    'Industrials': ['CAT','RTX','HON','GE','DE','ETN','NSC','GD','EMR','PCAR','CARR','CTAS'],
    'Consumer Staples': ['PG','KO','PEP','WMT','COST','MO','MDLZ','CL'],
    'Energy': ['XOM','CVX','EOG','VLO','PSX'],
    'Utilities': ['DUK','SO','PCG'],
    'Real Estate': ['PLD'],
    'Materials': ['SHW','MMM'],
}

sector_counts = pd.Series({k: len(v) for k, v in sector_assignments.items()}).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sector_counts.plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Number of Stocks')
axes[0].set_title('Universe Sector Breakdown\n(Top 100 S&P 500 by Market Cap)')
axes[0].axvline(sector_counts.mean(), color='red', linestyle='--', label=f'Mean: {sector_counts.mean():.1f}')
axes[0].legend()

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(sector_counts)))
axes[1].pie(sector_counts.values, labels=sector_counts.index, colors=colors, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Sector Weight Distribution')

plt.suptitle('Figure 0: S&P 500 Top 100 Universe — Sector Composition', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'universe_sector_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nTechnology dominance: IT = {sector_counts["Information Technology"]/sector_counts.sum()*100:.0f}% of universe')

---
## Section 2: Factor Portfolio Construction

We implement three factors using price data only:
1. **Momentum (12-1)**: 12-month return skipping the most recent month
2. **Low Volatility**: Negative of 63-day realized volatility (low vol = high score)
3. **Quality (proxy)**: Inverse of 252-day maximum drawdown magnitude

Each factor is cross-sectionally ranked; the top quintile (top 20% of names) forms the factor portfolio.

In [ ]:
# Generate synthetic price data for demonstration
# In production: price_df = data_loader.fetch_price_data(UNIVERSE, config.BACKTEST_START, config.BACKTEST_END)

np.random.seed(42)
n_days = 3528   # ~14 years of trading days
n_stocks = len(UNIVERSE)

dates = pd.bdate_range(start=config.BACKTEST_START, periods=n_days)

# Simulate stock prices with sector correlations and factor exposures
# Market factor
market_returns = np.random.normal(0.0003, 0.012, n_days)
# Individual stock returns: market beta + idiosyncratic
betas = np.random.uniform(0.6, 1.4, n_stocks)
idio = np.random.normal(0, 0.015, (n_days, n_stocks))
stock_returns = betas * market_returns[:, np.newaxis] + idio

# Build price series
price_data = pd.DataFrame(
    np.cumprod(1 + stock_returns, axis=0) * 100,
    index=dates,
    columns=UNIVERSE
)

print(f'Price data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} → {price_data.index[-1].date()}')
print(f'\nSample (first 5 rows, first 5 cols):')
print(price_data.iloc[:5, :5].round(2))

In [ ]:
# Compute all three factors
print('Computing factors...')
momentum_scores = factor_engine.compute_momentum_factor(price_data)
low_vol_scores = factor_engine.compute_low_vol_factor(price_data)
quality_scores = factor_engine.compute_quality_factor(price_data)

print(f'\nFactor score shapes: {momentum_scores.shape}')
print(f'Momentum NaN rate: {momentum_scores.isna().mean().mean():.1%}')
print(f'Low-vol NaN rate: {low_vol_scores.isna().mean().mean():.1%}')
print(f'Quality NaN rate: {quality_scores.isna().mean().mean():.1%}')

In [ ]:
# Plot factor score distributions
sample_date = dates[-252]  # About 1 year from end

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

factor_data = [
    (momentum_scores.loc[sample_date].dropna(), 'Momentum (12-1)', 'steelblue'),
    (low_vol_scores.loc[sample_date].dropna(), 'Low Volatility Score', 'forestgreen'),
    (quality_scores.loc[sample_date].dropna(), 'Quality Proxy Score', 'darkorange'),
]

for ax, (scores, title, color) in zip(axes, factor_data):
    ax.hist(scores, bins=20, color=color, alpha=0.7, edgecolor='white')
    q80 = scores.quantile(0.80)
    ax.axvline(q80, color='red', linestyle='--', linewidth=2, label=f'80th pct: {q80:.3f}')
    ax.set_title(f'{title}\nScore Distribution on {sample_date.date()}')
    ax.set_xlabel('Factor Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Figure 1: Cross-Sectional Factor Score Distributions\n(Red dashed = top-quintile threshold)', 
             fontweight='bold')
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'factor_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build factor portfolios
print('Building factor portfolios (monthly rebalance, top quintile)...')
momentum_portfolio = factor_engine.build_factor_portfolio(momentum_scores, 'momentum')
print(f'Momentum portfolio: {momentum_portfolio["date"].nunique()} rebalance dates')
print(f'Average holdings: {momentum_portfolio.groupby("date").size().mean():.1f} stocks')
print(f'\nSample holdings on first rebalance:')
first_date = momentum_portfolio['date'].iloc[0]
print(momentum_portfolio[momentum_portfolio['date'] == first_date][['ticker', 'factor_score', 'factor_rank', 'portfolio_weight']].head(10).to_string(index=False))

In [ ]:
# Turnover analysis
rebalance_dates = sorted(momentum_portfolio['date'].unique())
turnover_rates = []

for i in range(1, len(rebalance_dates)):
    prev = set(momentum_portfolio[momentum_portfolio['date'] == rebalance_dates[i-1]]['ticker'])
    curr = set(momentum_portfolio[momentum_portfolio['date'] == rebalance_dates[i]]['ticker'])
    turnover = len(curr - prev) / len(curr) if curr else 0
    turnover_rates.append({'date': rebalance_dates[i], 'turnover': turnover})

turnover_df = pd.DataFrame(turnover_rates).set_index('date')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(turnover_df.index, turnover_df['turnover'] * 100, color='steelblue', alpha=0.7, width=20)
ax.axhline(turnover_df['turnover'].mean() * 100, color='red', linestyle='--', 
           label=f'Average: {turnover_df["turnover"].mean()*100:.1f}%')
ax.set_xlabel('Rebalance Date')
ax.set_ylabel('One-Way Turnover (%)')
ax.set_title('Figure 2: Monthly Momentum Portfolio Turnover\n(New names entering the top quintile)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'portfolio_turnover.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAverage monthly turnover: {turnover_df["turnover"].mean()*100:.1f}%')
print(f'Transaction cost at {config.TRANSACTION_COST_BPS} bps: {turnover_df["turnover"].mean()*config.TRANSACTION_COST_BPS/100:.1f} bps per rebalance')

---
## Section 3: Options Skew Data Overview

The put skew is defined as:

$$\text{Put Skew} = IV_{25\Delta\text{ put}} - IV_{\text{ATM put}}$$

A positive value is the normal state (OTM puts trade at a premium to ATM). An unusually high put skew — measured as a z-score relative to recent history — signals elevated demand for downside protection.

**Data source note:** In this notebook we use synthetic skew data to demonstrate the methodology. In production, use `data_loader.fetch_options_skew_barchart()` for each ticker and date, or a paid options data feed (CBOE LiveVol, OptionMetrics, Interactive Brokers).

In [ ]:
# Generate synthetic put skew data
# Real skew = mean ~5% (5 vol points), std ~2%, with occasional spikes during stress

np.random.seed(42)

# Base skew: slowly mean-reverting
def generate_skew_series(n_days, base_level=0.05, stress_dates=None, seed=None):
    rng = np.random.default_rng(seed)
    skew = np.zeros(n_days)
    skew[0] = base_level
    for t in range(1, n_days):
        # AR(1) process
        skew[t] = 0.85 * skew[t-1] + 0.15 * base_level + rng.normal(0, 0.008)
    
    # Inject stress spikes
    if stress_dates:
        for d in stress_dates:
            if 0 < d < n_days - 30:
                skew[d:d+30] += 0.08 * np.exp(-np.arange(30) / 10)  # Spike that decays
    
    return np.maximum(skew, 0.005)  # Skew can't be negative (arbitrage)

# Stress periods aligned with known crash dates
stress_idx = []
for crash_date_str in ['2018-02-05', '2020-03-16', '2020-09-04', '2021-11-26']:
    crash_ts = pd.Timestamp(crash_date_str)
    if dates.min() <= crash_ts <= dates.max():
        idx = dates.get_loc(crash_ts, method='nearest')
        stress_idx.append(max(0, idx - 15))  # Start stress 15 days before crash

sample_tickers = ['AAPL', 'MSFT', 'AMZN', 'JPM', 'NVDA', 'GOOGL']
skew_panel_raw = {}
for i, ticker in enumerate(UNIVERSE):
    ticker_stress = stress_idx[i % len(stress_idx)] if stress_idx else []
    skew_panel_raw[ticker] = generate_skew_series(n_days, base_level=0.04+0.02*np.random.rand(),
                                                   stress_dates=stress_idx, seed=i)

skew_panel = pd.DataFrame(skew_panel_raw, index=dates)
print(f'Skew panel shape: {skew_panel.shape}')
print(f'Average put skew: {skew_panel.mean().mean():.1%} (5-10% is normal for large caps)')

In [ ]:
# Plot put skew time series for 3 sample tickers
sample_tickers_plot = ['AAPL', 'MSFT', 'JPM']

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, ticker in zip(axes, sample_tickers_plot):
    ax.plot(dates, skew_panel[ticker] * 100, color='steelblue', linewidth=0.8, label=ticker)
    rolling_mean = skew_panel[ticker].rolling(20).mean()
    ax.plot(dates, rolling_mean * 100, color='red', linewidth=1.5, alpha=0.8, label='20-day MA')
    ax.set_ylabel('Put Skew (vol pts)', fontsize=10)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_ylim(0, None)
    
    # Mark crash dates
    for crash_date_str in list(config.FACTOR_CRASH_DATES.values()):
        crash_ts = pd.Timestamp(crash_date_str)
        if dates.min() <= crash_ts <= dates.max():
            ax.axvline(crash_ts, color='red', alpha=0.4, linewidth=1, linestyle='--')

axes[0].set_title('Figure 3: Put Skew Time Series for Sample Tickers\n(Red dashed lines = known factor crash dates)', 
                   fontweight='bold')
axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'skew_time_series_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of put skew values across universe
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of all skew values
all_skew = skew_panel.values.flatten()
all_skew = all_skew[~np.isnan(all_skew)] * 100

axes[0].hist(all_skew, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(np.percentile(all_skew, 75), color='orange', linestyle='--', label='75th pct')
axes[0].axvline(np.percentile(all_skew, 90), color='red', linestyle='--', label='90th pct')
axes[0].set_xlabel('Put Skew (vol pts)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Put Skew Values\n(All tickers, all dates)')
axes[0].legend()

# Coverage analysis (% of dates with valid skew per ticker)
coverage = skew_panel.notna().mean() * 100
axes[1].hist(coverage, bins=20, color='forestgreen', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Data Coverage (%)')
axes[1].set_ylabel('Number of Tickers')
axes[1].set_title(f'Data Coverage by Ticker\nMean: {coverage.mean():.1f}%')

plt.suptitle('Figure 4: Skew Data Quality Assessment', fontweight='bold')
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'skew_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Universe mean put skew: {all_skew.mean():.2f} vol pts')
print(f'Universe std put skew: {all_skew.std():.2f} vol pts')
print(f'Average coverage: {coverage.mean():.1f}%')

---
## Section 4: Skew Z-Score Construction

The raw put skew level varies across stocks (AAPL tends to have a persistently higher skew than, say, a utility stock). To make the signal comparable across names, we compute a rolling z-score:

$$z_{i,t} = \frac{\text{skew}_{i,t} - \overline{\text{skew}}_{i,[t-20, t]}}{\sigma_{i,[t-20, t]}}$$

Then **winsorize at ±4** to clip extreme values from data errors.

After z-scoring, all tickers are on the same scale: 0 = average recent skew, +2 = unusually elevated.

In [ ]:
# Compute rolling z-scores
from src.utils import rolling_zscore

skew_zscore_panel = pd.DataFrame(index=dates, columns=UNIVERSE, dtype=float)
for col in UNIVERSE:
    skew_zscore_panel[col] = rolling_zscore(
        skew_panel[col], 
        window=config.SKEW_ROLLING_WINDOW,
        winsorise_clip=config.SKEW_WINSOR_CLIP
    )

print(f'Z-score panel: {skew_zscore_panel.shape}')
print(f'Z-score mean: {skew_zscore_panel.mean().mean():.3f} (should ≈ 0)')
print(f'Z-score std: {skew_zscore_panel.std().mean():.3f} (should ≈ 1)')
print(f'Values clipped at ±{config.SKEW_WINSOR_CLIP}: {(skew_zscore_panel.abs() == config.SKEW_WINSOR_CLIP).sum().sum()} cells')

In [ ]:
# Show raw skew vs z-score side by side for AAPL
ticker = 'AAPL'
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Raw skew
ax1.plot(dates, skew_panel[ticker] * 100, color='steelblue', linewidth=0.8)
ax1.set_ylabel('Raw Put Skew (vol pts)', fontsize=10)
ax1.set_title(f'Figure 5: {ticker} — Raw Put Skew vs Rolling Z-Score', fontweight='bold')

# Z-score
zscore = skew_zscore_panel[ticker]
ax2.plot(dates, zscore, color='darkorange', linewidth=0.8)
ax2.axhline(config.CROWDING_THRESHOLD, color='red', linestyle='--', linewidth=1.5, 
            label=f'WARNING threshold (+{config.CROWDING_THRESHOLD})')
ax2.axhline(-config.CROWDING_THRESHOLD, color='blue', linestyle='--', linewidth=1.5,
            label=f'DEPRESSED threshold (-{config.CROWDING_THRESHOLD})')
ax2.axhline(0, color='grey', linewidth=0.5, linestyle=':')
ax2.fill_between(dates, config.CROWDING_THRESHOLD, zscore, 
                  where=(zscore > config.CROWDING_THRESHOLD), alpha=0.3, color='red', label='Elevated skew')
ax2.set_ylabel('Rolling Z-Score (20-day)', fontsize=10)
ax2.set_xlabel('Date')
ax2.legend(fontsize=9, loc='upper left')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / f'zscore_vs_raw_{ticker}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5: Factor Crowding Index Construction

The crowding index aggregates single-name skew signals into a portfolio-level metric:

$$\text{CrowdingIndex}_{t} = \text{median}\left\{ z_{i,t} \mid i \in \text{TopQuintile}_t \right\}$$

**Why median?** The median is robust to outliers. A single stock's options being mispriced (stale quotes, corporate action) should not trigger a market-wide warning. The median requires the crowding signal to be broad-based.

In [ ]:
# Compute crowding index using the module
crowding_index = crowding_detector.compute_factor_crowding_index(
    skew_zscore_panel, momentum_portfolio, 'momentum'
)

print(f'Crowding index: {crowding_index.notna().sum()} valid observations')
print(f'Mean: {crowding_index.mean():.3f}')
print(f'Std: {crowding_index.std():.3f}')
print(f'Max: {crowding_index.max():.3f} on {crowding_index.idxmax().date()}')
print(f'Min: {crowding_index.min():.3f} on {crowding_index.idxmin().date()}')

In [ ]:
# Plot crowding index with crash dates annotated
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(crowding_index.index, crowding_index.values, color='#1a1a2e', linewidth=1, alpha=0.8)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

# Shade periods above threshold
ax.fill_between(crowding_index.index, config.CROWDING_THRESHOLD, crowding_index.values,
                where=(crowding_index.values > config.CROWDING_THRESHOLD),
                alpha=0.4, color='red', label=f'WARNING (>{config.CROWDING_THRESHOLD})')

# Crash date lines
for event_name, date_str in config.FACTOR_CRASH_DATES.items():
    crash_ts = pd.Timestamp(date_str)
    if crowding_index.index.min() <= crash_ts <= crowding_index.index.max():
        ax.axvline(crash_ts, color='red', linewidth=1.5, linestyle=':', alpha=0.7)
        ax.annotate(date_str[:7], xy=(crash_ts, crowding_index.max()*0.85),
                    rotation=90, fontsize=7, color='darkred')

ax.axhline(config.CROWDING_THRESHOLD, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Crowding Index (z-score)')
ax.set_title('Figure 6: Momentum Factor Crowding Index\nCross-sectional median put-skew z-score of top-quintile holdings',
             fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'crowding_index_momentum.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-sectional skew distribution: normal day vs WARNING regime day
crowding_df = crowding_detector.detect_crowding_regime(crowding_index)

warning_dates = crowding_df[crowding_df['regime_label'] == config.REGIME_WARNING].index
normal_dates = crowding_df[crowding_df['regime_label'] == config.REGIME_NORMAL].index

if len(warning_dates) > 0 and len(normal_dates) > 0:
    # Sample one date from each regime
    sample_warning = warning_dates[len(warning_dates)//2]
    sample_normal = normal_dates[len(normal_dates)//2]
    
    # Get factor holdings on those dates
    portfolio_tickers = list(momentum_portfolio['ticker'].unique()[:20])
    available = [t for t in portfolio_tickers if t in skew_zscore_panel.columns]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    
    z_normal = skew_zscore_panel.loc[sample_normal, available].dropna()
    z_warning = skew_zscore_panel.loc[sample_warning, available].dropna()
    
    ax1.hist(z_normal, bins=15, color='steelblue', alpha=0.8, edgecolor='white')
    ax1.axvline(z_normal.median(), color='red', linewidth=2, label=f'Median: {z_normal.median():.2f}')
    ax1.set_title(f'NORMAL Regime\n{sample_normal.date()}')
    ax1.set_xlabel('Skew Z-Score')
    ax1.set_ylabel('Count')
    ax1.legend()
    
    ax2.hist(z_warning, bins=15, color='#cc3300', alpha=0.8, edgecolor='white')
    ax2.axvline(z_warning.median(), color='darkred', linewidth=2, label=f'Median: {z_warning.median():.2f}')
    ax2.set_title(f'WARNING Regime\n{sample_warning.date()}')
    ax2.set_xlabel('Skew Z-Score')
    ax2.legend()
    
    plt.suptitle('Figure 7: Skew Z-Score Distribution — Normal vs WARNING Regime\n'
                 'WARNING regime shows rightward shift: factor names are being hedged',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / 'skew_distribution_regime_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient WARNING/NORMAL regime periods for comparison (check synthetic data range)')

---
## Section 6: Lead-Lag Analysis

If the crowding index is predictive, it should be negatively correlated with **future** factor returns at positive lags.

- **Lag 0**: Coincident (not actionable)
- **Lag 5-10**: Short-term predictability (most actionable)
- **Lag 10-20**: Longer-term (noisier)

We compute the Pearson correlation between `CrowdingIndex[t]` and `FactorReturn[t+k]` for k=0,...,20.

In [ ]:
# Compute factor returns first
factor_returns = factor_engine.compute_factor_returns(momentum_portfolio, price_data)
print(f'Factor returns: {len(factor_returns)} periods')
print(f'Annualized return: {((1+factor_returns).prod()**(252/len(factor_returns))-1)*100:.1f}%')

# Lead-lag analysis
lag_df = crowding_detector.compute_crowding_lead_lag(crowding_df, factor_returns, max_lag=20)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#cc0000' if c < 0 else '#0066cc' for c in lag_df['correlation']]
ax.bar(lag_df['lag_days'], lag_df['correlation'], color=colors, alpha=0.8, width=0.7, edgecolor='white')

# Peak negative bar
min_idx = lag_df['correlation'].idxmin()
ax.bar(lag_df.loc[min_idx, 'lag_days'], lag_df.loc[min_idx, 'correlation'], color='#8b0000', width=0.7)
ax.annotate(f"Peak: lag={lag_df.loc[min_idx,'lag_days']}d\nr={lag_df.loc[min_idx,'correlation']:.3f}",
            xy=(lag_df.loc[min_idx,'lag_days'], lag_df.loc[min_idx,'correlation']),
            xytext=(lag_df.loc[min_idx,'lag_days']+1.5, lag_df.loc[min_idx,'correlation']-0.02),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10, color='#8b0000')

ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(2/np.sqrt(200), color='grey', linestyle='--', linewidth=1, alpha=0.7, label='Approx. p=0.05')
ax.axhline(-2/np.sqrt(200), color='grey', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Lag (days ahead)', fontsize=11)
ax.set_ylabel('Pearson Correlation', fontsize=11)
ax.set_title('Figure 8: Lead-Lag Correlation — Crowding Index vs Future Factor Returns\n'
             'Negative correlation at positive lags = crowding predicts underperformance', fontweight='bold')
ax.set_xticks(lag_df['lag_days'])
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'lead_lag_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

peak_lag = lag_df.loc[min_idx, 'lag_days']
peak_corr = lag_df.loc[min_idx, 'correlation']
print(f'\nPeak negative correlation: r={peak_corr:.3f} at lag={peak_lag} days')
print(f'p-value at peak lag: {lag_df.loc[min_idx, "p_value"]:.3f}')
if peak_lag >= 5:
    print(f'→ Signal is PREDICTIVE with {peak_lag}-day lead time (actionable)')
else:
    print(f'→ Signal is mostly coincident (add confirmation filter)')

---
## Section 7: Regime Classification

The crowding index is itself z-scored against its 60-day rolling history to produce the **Regime Z-Score**:

$$\text{RegimeZ}_t = \frac{\text{CrowdingIndex}_t - \overline{\text{CrowdingIndex}}_{[t-60,t]}}{\sigma_{[t-60,t]}}$$

Regime thresholds:
- **WARNING**: RegimeZ > +1.5 → Reduce exposure to 50%
- **ELEVATED**: +0.75 < RegimeZ ≤ +1.5 → Reduce exposure to 75%  
- **NORMAL**: −0.75 < RegimeZ ≤ +0.75 → Full exposure (100%)
- **DEPRESSED**: RegimeZ ≤ −0.75 → Mild leverage (110%, experimental)

In [ ]:
# Regime classification already computed above, display breakdown
regime_counts = crowding_df['regime_label'].value_counts()
regime_pct = (regime_counts / len(crowding_df) * 100).round(1)

print('Regime Distribution:')
print('=' * 40)
for regime in [config.REGIME_WARNING, config.REGIME_ELEVATED, config.REGIME_NORMAL, config.REGIME_DEPRESSED]:
    n = regime_counts.get(regime, 0)
    pct = regime_pct.get(regime, 0)
    exposure = config.REGIME_EXPOSURE_MAP[regime] * 100
    print(f'  {regime:12s}: {n:4d} days ({pct:5.1f}%) → {exposure:.0f}% factor exposure')

print(f'  Total: {len(crowding_df)} days')

In [ ]:
# Regime time series visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: Crowding index with regime colors
ax = axes[0]
regime_colors = {
    config.REGIME_WARNING: '#ffcccc',
    config.REGIME_ELEVATED: '#ffe5cc',
    config.REGIME_DEPRESSED: '#cce5ff',
}
for regime, color in regime_colors.items():
    mask = crowding_df['regime_label'] == regime
    ax.fill_between(crowding_df.index, crowding_df['regime_zscore'].min()-0.5,
                    crowding_df['regime_zscore'].max()+0.5,
                    where=mask, color=color, alpha=0.6)

ax.plot(crowding_df.index, crowding_df['regime_zscore'], color='#1a1a2e', linewidth=0.8)
ax.axhline(config.CROWDING_THRESHOLD, color='red', linestyle='--', linewidth=1, label=f'WARNING threshold (+{config.CROWDING_THRESHOLD})')
ax.axhline(config.ELEVATED_THRESHOLD, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(config.DEPRESSED_THRESHOLD, color='blue', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')
ax.set_ylabel('Regime Z-Score', fontsize=10)
ax.set_title('Figure 9: Regime Classification Over Time', fontweight='bold')
ax.legend(fontsize=9, loc='upper left')

# Panel 2: Regime as categorical color strip
ax2 = axes[1]
regime_numeric = crowding_df['regime_label'].map({
    config.REGIME_WARNING: 3, config.REGIME_ELEVATED: 2,
    config.REGIME_NORMAL: 1, config.REGIME_DEPRESSED: 0
})

cmap = plt.get_cmap('RdYlBu_r', 4)
ax2.scatter(crowding_df.index, [1]*len(crowding_df), c=regime_numeric, cmap=cmap, 
             vmin=-0.5, vmax=3.5, s=1, marker='|')
ax2.set_yticks([1])
ax2.set_yticklabels(['Regime'])
ax2.set_xlabel('Date')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Legend
import matplotlib.patches as mpatches
patches = [
    mpatches.Patch(color=cmap(3/3), label='WARNING'),
    mpatches.Patch(color=cmap(2/3), label='ELEVATED'),
    mpatches.Patch(color=cmap(1/3), label='NORMAL'),
    mpatches.Patch(color=cmap(0/3), label='DEPRESSED'),
]
ax2.legend(handles=patches, loc='upper right', fontsize=9, ncol=4)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'regime_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Regime transition matrix
from src.regime_filter import compute_regime_transition_matrix
transition_matrix = compute_regime_transition_matrix(crowding_df)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(transition_matrix, ax=ax, cmap='Blues', vmin=0, vmax=1,
            annot=True, fmt='.2f', linewidths=1, linecolor='white',
            cbar_kws={'label': 'Transition Probability'},
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Figure 10: Regime Transition Probability Matrix\nP(tomorrow | today) — diagonal = persistence',
             fontweight='bold')
ax.set_xlabel('To Regime (Tomorrow)')
ax.set_ylabel('From Regime (Today)')
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'regime_transition_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Persistence stats
print('Regime Persistence (P(same → same)):')
for regime in transition_matrix.index:
    if regime in transition_matrix.columns:
        p = transition_matrix.loc[regime, regime]
        actionable = 'STICKY (act on first signal)' if p > 0.7 else 'TRANSIENT (require 3-day confirmation)'
        print(f'  {regime:12s}: {p:.1%} — {actionable}')

---
## Section 8: Crash Date Validation

This is the empirical heart of the research. For each known factor crash event, we ask:
1. Was the crowding index in WARNING regime in the 30 days before the crash?
2. How many days of lead time did the signal provide?
3. How many false positive WARNINGs appeared in the 60 days before the pre-crash window?

A good signal: WARNING triggered, meaningful lead time, low false positive rate.

In [ ]:
# Load crash dates reference
crash_dates_df = pd.read_csv(config.FACTOR_CRASH_DATES_CSV)

# Run validation
validation_df = crowding_detector.validate_against_crash_dates(crowding_df, crash_dates_df)

print('\nCROWDING SIGNAL VALIDATION AGAINST HISTORICAL FACTOR CRASHES')
print('=' * 80)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
display_df = validation_df[['event_name', 'warning_triggered', 'days_lead_time',
                              'peak_crowding_level', 'false_signal_count_prior_30d', 'regime_on_crash_date']]
print(display_df.to_string(index=False))

n_events = len(validation_df)
n_warned = validation_df['warning_triggered'].sum()
avg_lead = validation_df['days_lead_time'].dropna().mean()
avg_fp = validation_df['false_signal_count_prior_30d'].mean()

print(f'\nSummary:')
print(f'  Events with prior WARNING: {n_warned}/{n_events} ({n_warned/n_events*100:.0f}%)')
print(f'  Average lead time: {avg_lead:.1f} days')
print(f'  Average false signals per 60d: {avg_fp:.1f}')

---
## Section 9: Backtest Results

We compare two strategies:
1. **Baseline**: Factor portfolio at full exposure (100%) at all times
2. **Regime-Filtered**: Exposure scaled by crowding regime

**Look-ahead bias prevention**: The regime signal is lagged by 1 day before being applied to portfolio weights.

In [ ]:
# Build all tickers needed for defensive basket
all_tickers = UNIVERSE + config.DEFENSIVE_TICKERS
all_prices = price_data.copy()

# Add synthetic defensive ticker data
np.random.seed(123)
for def_ticker in config.DEFENSIVE_TICKERS:
    if def_ticker not in all_prices.columns:
        # Defensive: lower vol, lower beta, higher dividend (simulated)
        def_rets = np.random.normal(0.0002, 0.007, n_days)  # ~5% annual, 11% vol
        all_prices[def_ticker] = np.cumprod(1 + def_rets) * 100

# Apply regime filter
filtered_df = regime_filter.apply_regime_filter(factor_returns, crowding_df, all_prices)

# Run backtests
baseline_backtest = backtest.run_baseline_factor_backtest(factor_returns)
filtered_backtest = backtest.run_regime_filtered_backtest(filtered_df)

print('Backtest data shapes:')
print(f'  Baseline: {baseline_backtest.shape}')
print(f'  Filtered: {filtered_backtest.shape}')

In [ ]:
# Performance metrics
baseline_metrics = backtest.compute_performance_metrics(baseline_backtest, 'Baseline (Unfiltered)')
filtered_metrics = backtest.compute_performance_metrics(filtered_backtest, 'Regime-Filtered')
comparison = backtest.compare_strategies(baseline_metrics, filtered_metrics)

In [ ]:
# Cumulative returns comparison
fig, ax = plt.subplots(figsize=(14, 6))

# WARNING regime shading
if 'regime_label' in filtered_backtest.columns:
    mask = filtered_backtest['regime_label'] == config.REGIME_WARNING
    if mask.any():
        ax.fill_between(filtered_backtest.index, 0, filtered_backtest['cumulative_return'].max()*1.05,
                        where=mask, color='#ffcccc', alpha=0.4, label='WARNING regime')

ax.plot(baseline_backtest.index, baseline_backtest['cumulative_return'], 
         color='#888888', linewidth=2, linestyle='--', label=f'Baseline (Total: {baseline_metrics["total_return"]:+.1f}%)')
ax.plot(filtered_backtest.index, filtered_backtest['cumulative_return'],
         color='#0066cc', linewidth=2.5, label=f'Regime-Filtered (Total: {filtered_metrics["total_return"]:+.1f}%)')
ax.axhline(1.0, color='black', linewidth=0.5, linestyle=':')

# Crash date markers
for crash_date_str in list(config.FACTOR_CRASH_DATES.values()):
    crash_ts = pd.Timestamp(crash_date_str)
    if baseline_backtest.index.min() <= crash_ts <= baseline_backtest.index.max():
        ax.axvline(crash_ts, color='red', linewidth=0.8, linestyle=':', alpha=0.6)

# Sharpe ratios in legend
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return ($1 invested)')
ax.set_title(f'Figure 11: Cumulative Returns Comparison\n'
             f'Baseline Sharpe: {baseline_metrics["sharpe"]:.2f} | Filtered Sharpe: {filtered_metrics["sharpe"]:.2f}',
             fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / 'cumulative_returns_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Crash event study
event_study = backtest.run_crash_date_event_study(baseline_backtest, filtered_backtest, crash_dates_df)
print('\nCRASH EVENT STUDY — Capital Protection Analysis')
print('=' * 70)
print(event_study.to_string(index=False))

In [ ]:
# Event study chart
df_events = event_study.dropna(subset=['baseline_crash_return', 'filtered_crash_return'])

if not df_events.empty:
    x = np.arange(len(df_events))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    bars1 = ax.bar(x - width/2, df_events['baseline_crash_return'], width,
                    label='Baseline', color='#cc3300', alpha=0.8, edgecolor='white')
    bars2 = ax.bar(x + width/2, df_events['filtered_crash_return'], width,
                    label='Regime-Filtered', color='#ff8c00', alpha=0.8, edgecolor='white')

    # Protection annotations
    for i, (_, row) in enumerate(df_events.iterrows()):
        prot = row.get('protection_pct', 0)
        if not pd.isna(prot) and prot > 0:
            ax.text(i, max(row['baseline_crash_return'], row['filtered_crash_return']) + 0.3,
                    f'+{prot:.1f}%', ha='center', fontsize=9, color='darkgreen', fontweight='bold')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Crash Event')
    ax.set_ylabel('Return During Crash Window (%)')
    ax.set_title('Figure 12: Capital Protection During Factor Crashes\n'
                  'Green = protection provided by regime filter', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_events['event_name'], rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / 'crash_event_study.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No crash events fall within the backtest period [2010-2023].')

In [ ]:
# Scatter: crowding level vs future factor returns
from scipy import stats

# Compute 30-day forward returns
forward_window = 30
daily_factor_rets = factor_returns.reindex(crowding_df.index, method='ffill').fillna(0)
forward_ret = daily_factor_rets.rolling(window=forward_window).sum().shift(-forward_window)

plot_df = pd.concat([crowding_df['crowding_index'], crowding_df['regime_label'], forward_ret.rename('forward_return')], 
                     axis=1, join='inner').dropna()

if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors_map = {config.REGIME_WARNING: '#cc0000', config.REGIME_ELEVATED: '#ff8c00',
                  config.REGIME_NORMAL: '#888888', config.REGIME_DEPRESSED: '#0066cc'}
    
    for regime, color in colors_map.items():
        mask = plot_df['regime_label'] == regime
        if mask.sum() > 2:
            ax.scatter(plot_df.loc[mask, 'crowding_index'], plot_df.loc[mask, 'forward_return']*100,
                       c=color, alpha=0.4, s=15, label=regime)
    
    # OLS line
    x = plot_df['crowding_index'].values
    y = plot_df['forward_return'].values * 100
    slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope*x_line + intercept, 'k--', linewidth=2,
             label=f'OLS: β={slope:.3f}, R²={r_val**2:.3f}, p={p_val:.3f}')
    
    ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle=':')
    ax.set_xlabel('Crowding Index Level')
    ax.set_ylabel(f'Next {forward_window}-Day Factor Return (%)')
    ax.set_title(f'Figure 13: Crowding Index vs Forward {forward_window}-Day Returns\n'
                  f'Negative slope = crowding predicts underperformance (β={slope:.3f})', fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / 'scatter_crowding_vs_returns.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Section 10: Limitations & Extensions

### Known Limitations

**1. Options Data Quality**
Barchart delayed data introduces look-ahead bias in live use (20-min delay). In historical research, using today's options data to make today's portfolio decision is legitimate only if the data was observable before the trading window closed. For intraday decisions, use real-time data from a paid feed (CBOE LiveVol, OptionMetrics, IBKR).

**2. Delta Approximation**
Using the closest available strike to 25-delta is not a true 25-delta calculation. Accurate delta requires a vol surface and option pricing model (Black-Scholes or local vol). The moneyness heuristic (90-95% of spot) is a reasonable proxy for liquid options but degrades for names with skewed spot distributions.

**3. Universe Size**
100 names may not provide enough cross-sectional diversity for the median to be robust. With 20 top-quintile holdings, a single bad data point affects 5% of the median input. Production implementation should target 200+ names with a filter on options open interest.

**4. Quality Factor Proxy**
Price-based quality (inverse max drawdown) is a weak proxy for fundamental quality. The ROE/debt/earnings stability composite used by MSCI Quality Index requires accounting data. The current proxy will misclassify leveraged buyout stocks that happened to have smooth prices.

**5. Transaction Costs**
Reducing to 50% exposure and back involves trading half the portfolio. At 10 bps one-way, a round-trip regime change costs ~10 bps. High regime transition frequency (many WARNING↔NORMAL cycles) could erode performance meaningfully. A 3-day confirmation filter before acting on a regime change would reduce unnecessary trading.

### Extensions

**1. Sector ETF Options**
Single-name skew is noisy. Sector ETF options (XLK, XLF, etc.) aggregate the signal: XLK put skew elevation = technology sector being hedged. Cleaner signal, lower data cost, easier backtesting.

**2. Cross-Asset Crowding**
Credit skew (CDX IG vs HY spread differential) and currency options (risk reversals) provide additional crowding signals. Multi-asset crowding composite may be more robust than equity-only.

**3. Multi-Factor Crowding**
Build a composite crowding index across all three factors (momentum, low-vol, quality). Factor crowding is correlated — when one factor unwinds, others often follow. A composite signal may provide earlier warning.

**4. Regime Confirmation Filter**
Require WARNING regime on 3 consecutive days before reducing exposure. This reduces false positives at the cost of 2-3 days of lag. Given that the typical crash unfolds over weeks (not hours), this tradeoff is likely favorable.

**5. Dynamic Threshold Calibration**
The 1.5 z-score WARNING threshold is fixed. An adaptive threshold that accounts for the current VIX level (raise threshold when VIX is already elevated — spikes are common) may reduce false positives during high-vol regimes.

---
**Disclaimer:** This notebook is academic research only. Results use synthetic data for demonstration purposes. Do not use this as investment advice.

In [ ]:
# Summary statistics table
print('RESEARCH SUMMARY')
print('=' * 60)
print(f'Universe: Top {config.UNIVERSE_SIZE} S&P 500 by market cap')
print(f'Backtest: {config.BACKTEST_START} → {config.BACKTEST_END}')
print(f'Factors: {config.FACTOR_NAMES}')
print(f'\nKey Parameters:')
print(f'  Skew rolling window: {config.SKEW_ROLLING_WINDOW} days')
print(f'  Crowding baseline:   {config.SKEW_ZSCORE_HISTORY} days')
print(f'  Warning threshold:   {config.CROWDING_THRESHOLD} z-score')
print(f'  Warning exposure:    {config.WARNING_EXPOSURE*100:.0f}%')
print(f'  Transaction cost:    {config.TRANSACTION_COST_BPS} bps')
print(f'\nAll outputs saved to: {config.OUTPUTS_DIR}')

import os
outputs = list(config.OUTPUTS_DIR.glob('*.png'))
print(f'\nFigures generated: {len(outputs)}')
for f in sorted(outputs):
    print(f'  {f.name}')